[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C25_Long_Context_Course/01_rope_scaling/01_rope_scaling.ipynb)

# 01 · RoPE Scaling 与 YaRN（用 numpy 从零写出）

目标：把 **RoPE**、**Position Interpolation (PI)**、**NTK-aware**、**YaRN（分频段 ramp + 注意力温度）** 用 numpy 从零实现，
并用**波长**、**外推角度误差**、**相对位置点积曲线**量化对比它们的外推质量。

**路线**：
1. RoPE 从零（频率、旋转、点积只依赖相对位置）
2. 位置墙：低频维度外推进入未见区间
3. **PI**：等比压回训练范围
4. **NTK-aware**：改一个 base，高频不动、低频插值
5. **YaRN ramp**：按转圈数分频段插值
6. **YaRN 温度** + 长度泛化度量
7. ✏️ 练习（PI / NTK base / YaRN ramp / 外推误差）→ 📖 答案 → 🧪 真实 config 胶囊

> **本课纪律**：每个缩放都与朴素 RoPE 在**短位置**上对拍（缩放在训练长度内应几乎不改变行为），用 `np.allclose` 兜底。

## 1 · RoPE 从零：频率、旋转、相对性

RoPE 把 query/key 按维度**成对**旋转，第 `i` 对维度的频率 `θ_i = base^(-2i/d)`，位置 `m` 处旋转角 `m·θ_i`。
核心性质：旋转后两个向量的点积**只依赖相对位置** `m-n`。我们从零实现并验证这条性质。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def rope_freqs(d, base=10000.0):
    '''返回 d/2 个频率 θ_i = base^(-2i/d)，i=0..d/2-1。低维高频、高维低频。'''
    i = np.arange(d // 2)
    return base ** (-2.0 * i / d)            # (d/2,)

def apply_rope(x, pos, freqs):
    '''对向量/矩阵 x (..., d) 在位置 pos 处施加 RoPE 旋转。
       把维度两两配对 (x0,x1),(x2,x3),... 每对按角 pos*θ 旋转。'''
    d = x.shape[-1]
    ang = pos * freqs                         # (d/2,) 每对的旋转角
    cos, sin = np.cos(ang), np.sin(ang)
    x_even = x[..., 0::2]                      # (..., d/2)
    x_odd  = x[..., 1::2]
    out = np.empty_like(x)
    out[..., 0::2] = x_even * cos - x_odd * sin   # 旋转矩阵作用
    out[..., 1::2] = x_even * sin + x_odd * cos
    return out

d = 64
freqs = rope_freqs(d)
print('频率范围: θ_max(高频)=%.4f  θ_min(低频)=%.2e' % (freqs[0], freqs[-1]))
# 旋转保模长（正交变换）
q = rng.standard_normal(d)
qr = apply_rope(q, 5, freqs)
assert np.allclose(np.linalg.norm(q), np.linalg.norm(qr)), 'RoPE 旋转应保模长'
print('✅ RoPE 旋转保模长（是正交变换）')

**验证相对性**：`<RoPE(q, m), RoPE(k, n)>` 应只依赖 `m-n`。固定相对距离、平移绝对位置，点积应不变。

In [ ]:
q = rng.standard_normal(d); k = rng.standard_normal(d)
def rope_dot(q, k, m, n, freqs):
    return apply_rope(q, m, freqs) @ apply_rope(k, n, freqs)

# 相对距离都 = 3，但绝对位置不同 -> 点积应几乎相等
dots = [rope_dot(q, k, m, m-3, freqs) for m in [3, 10, 50, 100]]
print('相对距离=3, 不同绝对位置的点积:', np.round(dots, 6))
assert np.allclose(dots, dots[0], atol=1e-9), 'RoPE 点积应只依赖相对位置'
print('✅ RoPE 点积只依赖相对位置 m-n —— 这是它做相对位置编码的根据')

## 2 · 位置墙：低频维度外推进入「未见区间」

训练只到 `L_train`。对每对维度算它在训练末尾转了几圈 `r_i = L_train·θ_i/(2π)`。
高频维度转很多圈（角度区间被扫遍），低频维度转不满一圈（外推必入未见区）。

In [ ]:
L_train = 4096
def turns_at(L, freqs):
    '''每对维度在长度 L 内转的圈数 r_i = L*θ_i/(2π)。'''
    return L * freqs / (2 * np.pi)

r = turns_at(L_train, freqs)
print(f'训练长度 {L_train} 内：最高频维度转 {r[0]:.0f} 圈，最低频维度转 {r[-1]:.3f} 圈')
n_unsafe = int((r < 1.0).sum())          # 转不满一圈 = 外推风险维度
print(f'有 {n_unsafe}/{d//2} 对维度在训练长度内没转满 1 圈（外推风险区）')
assert r[0] > 100 and r[-1] < 1.0, '高频转很多圈、低频转不满一圈'
assert n_unsafe > 0, '应存在没转满圈的低频维度'
print('✅ 位置墙：低频维度是外推的薄弱环节 —— 所有 scaling 都在安顿它们')

## 3 · Position Interpolation：等比压回训练范围

缩放因子 `s = L_target / L_train`。PI 把位置除以 `s`（等价于把每个频率除以 `s`，波长拉长 `s` 倍）。
**对拍**：在训练长度*以内*，PI 缩放后短距离的点积应与原始 RoPE 接近（没破坏近处）；但分辨率会变粗。

In [ ]:
def pi_freqs(freqs, s):
    '''Position Interpolation：所有频率均匀除以 s。'''
    return freqs / s

s = 4.0                                   # 4k -> 16k
freqs_pi = pi_freqs(freqs, s)
# 等价性：PI 下位置 m 的角 == 原始 RoPE 下位置 m/s 的角
m = 8000
ang_pi   = m * freqs_pi
ang_orig = (m / s) * freqs
assert np.allclose(ang_pi, ang_orig), 'PI(位置 m) 应等于 原始(位置 m/s)'
print('✅ PI 把位置 8000 的旋转角压成了原始位置 2000 的角（落回训练范围）')

# 波长全部拉长 s 倍
wav_orig = 2*np.pi/freqs
wav_pi   = 2*np.pi/freqs_pi
assert np.allclose(wav_pi, s*wav_orig), 'PI 把所有波长拉长 s 倍'
print(f'PI 把所有维度的波长统一拉长 {s:.0f} 倍 —— 包括本不需要的高频（伤局部分辨率）')

## 4 · NTK-aware：改一个 base，高频不动、低频插值

NTK 把 `base` 放大到 `base' = base · s^(d/(d-2))`，再重算频率。
效果：**高频维度几乎不变**（θ_0=1 完全不受 base 影响），**低频维度被插值**（且最低频恰好被插值 ~s 倍，与 PI 对齐）。

In [ ]:
def ntk_freqs(d, s, base=10000.0):
    '''NTK-aware：放大 base 后重算频率。'''
    base_new = base * (s ** (d / (d - 2)))
    return rope_freqs(d, base=base_new)

freqs_ntk = ntk_freqs(d, s)
# 高频端：NTK 几乎不动，PI 压了 s 倍
print('最高频 θ_0:  原始=%.4f  PI=%.4f  NTK=%.4f' % (freqs[0], freqs_pi[0], freqs_ntk[0]))
print('最低频 θ_-1: 原始=%.3e PI=%.3e NTK=%.3e' % (freqs[-1], freqs_pi[-1], freqs_ntk[-1]))
assert abs(freqs_ntk[0] - freqs[0]) < 0.05*freqs[0], 'NTK 高频应几乎不动'
assert freqs_pi[0] < 0.5*freqs[0], 'PI 把高频压了 s 倍'
# 低频端：NTK 与 PI 在最低频维度对齐（都插值 ~s 倍）
ratio = (freqs[-1] / freqs_ntk[-1])       # NTK 在最低频的插值倍数
print(f'NTK 在最低频维度插值了 {ratio:.2f} 倍（设计上 ≈ s={s:.0f}）')
assert abs(ratio - s) < 0.5*s, 'NTK 最低频插值倍数应接近 s'
print('✅ NTK：高频几乎不动（护局部分辨率），低频插值（敢外推）—— 比 PI 聪明')

## 5 · YaRN ramp：按转圈数分频段插值

YaRN 用每对维度的转圈数 `r_i = L_train·θ_i/(2π)` 和两个阈值 `α(=beta_slow), β(=beta_fast)` 算 ramp 权重：
$\gamma_i = \mathrm{clip}((r_i-\alpha)/(\beta-\alpha),0,1)$，再混合 `θ_i^YaRN = γ_i·θ_i + (1-γ_i)·θ_i/s`。
`γ=1`(高频,转>β圈)→不动；`γ=0`(低频,转<α圈)→完全插值(=PI)；中间平滑过渡。

In [ ]:
def yarn_ramp(freqs, L_train, alpha=1.0, beta=32.0):
    '''返回每对维度的 ramp 权重 γ_i ∈ [0,1]（1=保留原频率, 0=完全插值）。'''
    r = L_train * freqs / (2 * np.pi)        # 转圈数
    gamma = (r - alpha) / (beta - alpha)
    return np.clip(gamma, 0.0, 1.0)

def yarn_freqs(freqs, s, L_train, alpha=1.0, beta=32.0):
    gamma = yarn_ramp(freqs, L_train, alpha, beta)
    return gamma * freqs + (1 - gamma) * (freqs / s)

gamma = yarn_ramp(freqs, L_train)
freqs_yarn = yarn_freqs(freqs, s, L_train)
print('ramp γ: 高频端=%.2f ... 低频端=%.2f （应从 1 平滑降到 0）' % (gamma[0], gamma[-1]))
assert gamma[0] == 1.0 and gamma[-1] == 0.0, '高频 γ=1（不动），低频 γ=0（全插值）'
assert np.all(np.diff(gamma) <= 1e-12), 'γ 应随维度（频率降低）单调不增'
# 边界对齐：高频端 YaRN==原始RoPE，低频端 YaRN==PI
assert np.isclose(freqs_yarn[0], freqs[0]), '高频端 YaRN 应保留原频率'
assert np.isclose(freqs_yarn[-1], freqs_pi[-1]), '低频端 YaRN 应等于 PI'
print('✅ YaRN ramp：高频保原始、低频退化成 PI、中间平滑混合 —— 统一了前面所有方法')

## 6 · YaRN 温度 + 长度泛化度量

**温度**：插值抹平了注意力分布、升高了熵。YaRN 给 logits 乘 `1/t`（即旋转系数乘 `√(1/t)`）校正回去，
经验公式 `√(1/t) = 0.1·ln(s) + 1`。它是**位置无关的标量**，可吸收进 cos/sin，零额外开销。

**度量**：用「相对位置点积曲线」看外推质量——健康的位置编码应让点积随相对距离平滑变化。

In [ ]:
def yarn_temperature_scale(s):
    '''返回乘到 RoPE 旋转系数上的 √(1/t)（>1，把 logits 放大、分布变锐）。'''
    return 0.1 * np.log(s) + 1.0

ts = yarn_temperature_scale(s)
print(f's={s:.0f} -> 温度系数 √(1/t) = {ts:.4f}（>1：把被插值抹平的注意力分布重新调锐）')
assert ts > 1.0, 's>1 时温度系数应 >1'
assert abs(yarn_temperature_scale(1.0) - 1.0) < 1e-12, 's=1（不外推）时温度=1'

# 长度泛化度量：缩放后，超长位置的旋转角是否落回训练见过的角度范围
def max_angle_seen(L_train, freqs):
    '''训练时每对维度见过的最大旋转角（取低频维度最敏感）。'''
    return L_train * freqs[-1]              # 最低频维度的最大角

L_target = 16384
ang_train = max_angle_seen(L_train, freqs)
ang_naive = L_target * freqs[-1]            # 朴素外推：低频角冲出训练范围
ang_pi_   = L_target * freqs_pi[-1]         # PI：压回
print(f'\n最低频维度的最大旋转角：训练见过 {ang_train:.3f}')
print(f'  朴素外推到 16k = {ang_naive:.3f}  ({ang_naive/ang_train:.1f}x 训练范围，闯入未见区！)')
print(f'  PI   外推到 16k = {ang_pi_:.3f}   (≈训练范围，落回安全区)')
assert ang_naive > ang_train, '朴素外推冲出训练角度范围'
assert abs(ang_pi_ - ang_train) < 1e-6, 'PI 把外推角压回训练范围'
print('✅ 度量证实：朴素外推闯入未见角度区，PI/NTK/YaRN 把它拉回 → 这是外推不崩的根据')

---
## ✏️ 练习 1：实现 Position Interpolation 并验证落回范围

实现 `pi_scaled_pos(positions, s)`：把一串位置按 PI 等比压缩（除以 `s`）。
目标：压缩后，`L_target` 处的有效位置应 ≤ `L_train`（落回训练范围）。

In [ ]:
def pi_scaled_pos(positions, s):
    # TODO: 返回 positions / s（PI 的核心：位置等比压缩）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
L_train, L_target = 4096, 16384
s_ex = L_target / L_train
pos = np.array([0, 1000, 4096, 8192, 16384], dtype=float)
scaled = pi_scaled_pos(pos, s_ex)
assert np.allclose(scaled, pos / s_ex)
assert scaled[-1] <= L_train + 1e-6, 'L_target 处应被压回 <= L_train'
assert scaled[0] == 0.0, '位置 0 仍是 0'
print('压缩后的有效位置:', scaled)
print('✅ 练习 1 通过：PI 把 16384 压回到', scaled[-1], '（= L_train，落回训练范围）')

## ✏️ 练习 2：实现 NTK-aware 的 base 缩放

实现 `ntk_base(base, s, d)`：返回 NTK-aware 放大后的 base = `base · s^(d/(d-2))`。
再验证：用新 base 算出的频率，**高频几乎不动、最低频被插值约 s 倍**。

In [ ]:
def ntk_base(base, s, d):
    # TODO: 返回 base * s**(d/(d-2))
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
d_ex, s_ex = 64, 8.0
base_new = ntk_base(10000.0, s_ex, d_ex)
assert base_new > 10000.0, 'NTK 应放大 base'
f0 = rope_freqs(d_ex, 10000.0); f1 = rope_freqs(d_ex, base_new)
assert abs(f1[0] - f0[0]) < 0.05 * f0[0], '最高频几乎不变（θ_0=1 不受 base 影响）'
interp = f0[-1] / f1[-1]
assert abs(interp - s_ex) < 0.5 * s_ex, f'最低频应插值约 s={s_ex} 倍, 实得 {interp:.2f}'
print(f'NTK base: 10000 -> {base_new:.0f}；最低频插值 {interp:.2f} 倍（≈s={s_ex:.0f}）')
print('✅ 练习 2 通过：改一个 base 实现了高频不动、低频插值')

## ✏️ 练习 3：实现 YaRN ramp 的分频段权重

实现 `ramp_weight(r, alpha, beta)`：给定转圈数数组 `r`，返回 ramp 权重 `γ = clip((r-α)/(β-α), 0, 1)`。
目标：`r<=α` → γ=0（完全插值）；`r>=β` → γ=1（不动）；中间线性过渡且单调。

In [ ]:
def ramp_weight(r, alpha=1.0, beta=32.0):
    # TODO: 返回 clip((r-alpha)/(beta-alpha), 0, 1)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
r_test = np.array([0.5, 1.0, 16.5, 32.0, 100.0])
g = ramp_weight(r_test)
assert g[0] == 0.0 and g[1] == 0.0, 'r<=α 应 γ=0（完全插值）'
assert g[3] == 1.0 and g[4] == 1.0, 'r>=β 应 γ=1（不动）'
assert abs(g[2] - 0.5) < 1e-9, 'r 在 α,β 中点应 γ=0.5'
assert np.all(np.diff(g) >= -1e-12), 'γ 应随 r 单调不减'
print('ramp 权重:', np.round(g, 3))
print('✅ 练习 3 通过：分频段 ramp 正确（低频全插值、高频不动、中间过渡）')

## ✏️ 练习 4：测量外推角度误差

定义「外推误差」= 缩放后超长位置的最低频旋转角，**超出训练见过最大角的程度**（超出为正、落回为 0）。
实现 `extrapolation_error(freqs_scaled, L_train, L_target, freqs_orig)`，并验证：朴素>0（崩），PI≈0（安全）。

In [ ]:
def extrapolation_error(freqs_scaled, L_train, L_target, freqs_orig):
    # TODO:
    #   ang_seen = L_train * freqs_orig[-1]          # 训练见过的最大角(最低频)
    #   ang_tgt  = L_target * freqs_scaled[-1]       # 缩放后外推到 L_target 的角
    #   返回 max(ang_tgt - ang_seen, 0.0)            # 超出训练范围的部分
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
Lt, Lg = 4096, 16384
f_orig = rope_freqs(64)
f_pi   = pi_freqs(f_orig, Lg/Lt)
err_naive = extrapolation_error(f_orig, Lt, Lg, f_orig)   # 朴素：不缩放
err_pi    = extrapolation_error(f_pi,   Lt, Lg, f_orig)
assert err_naive > 0.0, '朴素外推应有正误差（冲出训练范围）'
assert err_pi < 1e-6, 'PI 应把误差压到 ~0（落回训练范围）'
print(f'外推误差：朴素={err_naive:.3f}（崩）  PI={err_pi:.3e}（安全）')
print('✅ 练习 4 通过：用一个标量量化了「外推有没有闯入未见区」')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def pi_scaled_pos(positions, s):
    return positions / s

In [ ]:
# 练习 2 参考答案
def ntk_base(base, s, d):
    return base * (s ** (d / (d - 2)))

In [ ]:
# 练习 3 参考答案
def ramp_weight(r, alpha=1.0, beta=32.0):
    return np.clip((r - alpha) / (beta - alpha), 0.0, 1.0)

In [ ]:
# 练习 4 参考答案
def extrapolation_error(freqs_scaled, L_train, L_target, freqs_orig):
    ang_seen = L_train * freqs_orig[-1]
    ang_tgt  = L_target * freqs_scaled[-1]
    return max(ang_tgt - ang_seen, 0.0)

---
## 🧪 真实数据胶囊：复现真实模型的 rope_scaling 配置

下面是一些**真实开源模型**用过的 RoPE 扩窗配置（来自其 HuggingFace config）。
用你实现的 PI/NTK/YaRN，把这些配置「翻译」成实际的频率改动，并验证它们都把外推角压回了训练范围。

In [ ]:
# 真实模型的扩窗配置（约数，来自公开 config）
REAL_CONFIGS = [
    # (模型, 类型, 原始长度, 目标长度, factor)
    ('CodeLlama',      'linear(PI)', 4096,  16384, 4.0),
    ('Yi-200K',        'linear(PI)', 4096, 200000, 50.0),
    ('Qwen-dynamic',   'dynamic NTK',4096,  32768, 8.0),
    ('Nous-Yarn-128k', 'yarn',       4096, 131072, 32.0),
]
d = 64; f_orig = rope_freqs(d)
def extrap_err(f_scaled, L_train, L_target, f_orig):
    '''外推误差 = 缩放后最低频维度外推角 超出训练见过最大角的程度（落回为 0）。'''
    ang_seen = L_train  * f_orig[-1]
    ang_tgt  = L_target * f_scaled[-1]
    return max(ang_tgt - ang_seen, 0.0)

print(f"{'模型':<16}{'类型':<14}{'factor':>7}{'外推误差(朴素→缩放后)':>24}")
for name, typ, orig, tgt, factor in REAL_CONFIGS:
    if 'PI' in typ:
        f_scaled = pi_freqs(f_orig, factor)
    elif 'NTK' in typ:
        f_scaled = ntk_freqs(d, factor)
    else:                       # yarn
        f_scaled = yarn_freqs(f_orig, factor, orig)
    err_naive = extrap_err(f_orig,   orig, tgt, f_orig)
    err_done  = extrap_err(f_scaled, orig, tgt, f_orig)
    print(f'{name:<16}{typ:<14}{factor:>7.0f}   {err_naive:>10.1f} → {err_done:>8.3f}')
print('\n观察：factor 越大（如 Yi-200K 的 50×），朴素外推误差越夸张，越离不开缩放。')

**🧪 胶囊练习**：实现 `scaling_reduces_error(name, typ, orig, tgt, factor, d=64)`：
对一个真实配置，返回 `(朴素误差, 缩放后误差)`，并断言缩放后误差 < 朴素误差的 1%。

In [ ]:
def scaling_reduces_error(name, typ, orig, tgt, factor, d=64):
    f_orig = rope_freqs(d)
    # TODO: 按 typ 选 PI/NTK/YaRN 算 f_scaled，返回 (err_naive, err_scaled)
    raise NotImplementedError

In [ ]:
# 自测
for name, typ, orig, tgt, factor in REAL_CONFIGS:
    en, es = scaling_reduces_error(name, typ, orig, tgt, factor)
    assert es < 0.01 * en + 1e-6, f'{name}: 缩放后误差应远小于朴素'
print('✅ 胶囊练习通过：所有真实配置缩放后都把外推角压回了训练范围')

In [ ]:
# 📖 胶囊参考答案
def scaling_reduces_error(name, typ, orig, tgt, factor, d=64):
    f_orig = rope_freqs(d)
    if 'PI' in typ:    f_scaled = pi_freqs(f_orig, factor)
    elif 'NTK' in typ: f_scaled = ntk_freqs(d, factor)
    else:              f_scaled = yarn_freqs(f_orig, factor, orig)
    en = extrap_err(f_orig,   orig, tgt, f_orig)
    es = extrap_err(f_scaled, orig, tgt, f_orig)
    return en, es

### 小结
- **RoPE** 按维度成对旋转，点积只依赖相对位置；频率 `θ_i=base^(-2i/d)` 高维低频、低频维度波长超训练长度。
- **位置墙**：低频维度在训练长度内没转满圈，外推闯入未见角度区 → 注意力崩。
- **PI**：位置/频率均匀除以 s，压回训练范围，但一刀切伤高频局部分辨率。
- **NTK-aware**：放大一个 base，高频几乎不动、低频插值（最低频对齐 PI）—— 免微调外推。
- **YaRN** = 分频段 ramp（高频不动 / 低频 PI / 中间过渡）+ 注意力温度（校正插值带来的熵升高）。
- 度量：波长/转圈数定位风险维度，外推角度误差量化「有没有闯入未见区」，是真实评测（模块 05）前的快速筛子。

下一站：**模块 02 · FlashAttention 与 IO 复杂度** —— 位置不崩之后，下一道墙是 n×n 注意力矩阵爆显存。